In [6]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from  sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [7]:
train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_parquet("../data/test_data.parquet")

X_train = train_data.drop(["ID_Employe", "Satisfait"], axis=1).select_dtypes(include=["number"])
y_train = train_data["Satisfait"]

print(train_data.isnull().sum())

# correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
# print(correlations.head(10))
test_data.isnull().sum()

ID_Employe                                0
Age                                       0
Sexe                                      0
Departement                               0
Annees_Experience                         0
Niveau_Etudes                             0
Salaire_Mensuel_BIF                       0
Travail_A_Distance                        0
Heures_Supplementaires                    0
Heures_Formation                          0
Anciennete_Entreprise                     0
Nombre_Promotions                         0
Evaluation_Performance                    0
Equilibre_Vie_Travail                     0
Satisfaction_Salaire                      0
Nombre_Absences                           0
Satisfait                                 0
Departement_Administration                0
Departement_Commercial                    0
Departement_Finance                       0
Departement_Informatique                  0
Departement_Logistique                    0
Departement_Marketing           

ID_Employe                                0
Age                                       0
Sexe                                      0
Departement                               0
Annees_Experience                         0
Niveau_Etudes                             0
Salaire_Mensuel_BIF                       0
Travail_A_Distance                        0
Heures_Supplementaires                    0
Heures_Formation                          0
Anciennete_Entreprise                     0
Nombre_Promotions                         0
Evaluation_Performance                    0
Equilibre_Vie_Travail                     0
Satisfaction_Salaire                      0
Nombre_Absences                           0
Satisfait                                 0
Departement_Administration                0
Departement_Commercial                    0
Departement_Finance                       0
Departement_Informatique                  0
Departement_Logistique                    0
Departement_Marketing           

### Mise en niveau pour les données de test

In [8]:
# Les variables numériques
# feature_cols = ["AgeMaison", "Quartier_Target", "Indicateur_Confort", "Chambres_par_Superficie", "LoyerMensuel_Log1"]

# for col in ['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin']:
#     test_data[col + "_Bin"] = test_data[col].map({"Oui": 1, "Non": 0}).fillna(0).astype(int)
file_path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "data", "artefacts.joblib")
artefacts = joblib.load(file_path) 
# test_data["Quartier_Target"] = encoder.transform(test_data[["Quartier"]])[:, 0]
print(artefacts)
# test_data["LoyerMensuel_Log1"] = np.log1p(test_data["LoyerMensuel_BIF"])

# Séparation de X_test et y_test
X_test = test_data.drop(["ID_Employe", "Satisfait"], axis=1).select_dtypes(include=["number"])
y_test = test_data["Satisfait"]

X_test.isnull().sum()

{'ordinal_encoder': OrdinalEncoder(categories=[['Très insatisfait', 'Insatisfait', 'Neutre',
                            'Satisfait', 'Très satisfait'],
                           ['Très mauvais', 'Mauvais', 'Moyen', 'Bon',
                            'Excellent']]), 'one_hot_encoder': OneHotEncoder(handle_unknown='ignore', sparse_output=False), 'standard_scaler': StandardScaler()}


Age                                       0
Annees_Experience                         0
Salaire_Mensuel_BIF                       0
Heures_Supplementaires                    0
Heures_Formation                          0
Anciennete_Entreprise                     0
Nombre_Promotions                         0
Evaluation_Performance                    0
Equilibre_Vie_Travail                     0
Satisfaction_Salaire                      0
Nombre_Absences                           0
Departement_Administration                0
Departement_Commercial                    0
Departement_Finance                       0
Departement_Informatique                  0
Departement_Logistique                    0
Departement_Marketing                     0
Departement_Production                    0
Departement_Recherche et Développement    0
Departement_Ressources Humaines           0
Departement_Support Client                0
dtype: int64

### Entrainement du modèle

In [9]:
random_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    n_jobs=-1
)

random_model.fit(X_train, y_train)

y_predict = random_model.predict(X_test)

y_predict_series = pd.Series(y_predict, index=y_test.index)

print(y_predict_series)
print()
print(y_test)

compareson = pd.DataFrame({
    "Valeurs réelles": y_test,
    "Valeurs prédites": y_predict_series
})

print(f1_score(y_test, y_predict))

compareson.head(10)


0      0
1      1
2      1
3      0
4      0
      ..
123    1
124    0
125    1
126    1
127    0
Length: 128, dtype: int64

0      0
1      1
2      0
3      1
4      1
      ..
123    1
124    0
125    1
126    0
127    0
Name: Satisfait, Length: 128, dtype: int64
0.8333333333333334


,Valeurs réelles,Valeurs prédites
0,0,0
1,1,1
2,0,1
3,1,0
4,1,0
5,1,1
6,1,0
7,1,1
8,1,1
9,1,1


### Sauvegarde du modèle

In [10]:

mean = {
    col: X_train[col].mean() for col in X_train.select_dtypes(include=['number']).columns}
modes = {
    col: X_train[col].mode()[0] for col in X_train.select_dtypes(include=['object']).columns}

artefacts = {
    'random_model': random_model,
    'one_hot_encoder': artefacts["one_hot_encoder"],
    'ordinal_encoder': artefacts["ordinal_encoder"],
    'standard_scaler': artefacts["standard_scaler"],
    'mean': mean,
    'modes': modes,
    'columns': list(X_train.columns)
}

path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "models", "best_model_tuned.pkl")
joblib.dump(artefacts, path)

print(f"Modèle et prétraitements sauvegardés dans {path}")

Modèle et prétraitements sauvegardés dans /home/luckson/AI-Project/employee_satisfaction/models/best_model_tuned.pkl
